# Lesson 200 · Year 5 exit exam

PROVIDED explains and supplies source. TODO implements three live contracts. CHECK calls your functions. EXIT writes a proposal for teacher review. The default replays all raw new L200 predictions; a fresh model run is a separate optional track. Complete narrative and portable architecture follow. Runtime budget: 3600s aggregate local work; package installation counts. No paid cloud calls in the default notebook.

In [1]:
# @colab-bootstrap
import ast,base64,hashlib,io,json,os,sys,tempfile,time,zipfile,subprocess,importlib.util,shutil
from pathlib import Path
started=time.monotonic()
for package in ["numpy","torch"]:
    if importlib.util.find_spec(package) is None:
        subprocess.run([sys.executable,"-m","pip","install",package],check=True,timeout=300)
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
torch.set_num_threads(1)
workspace=Path(tempfile.mkdtemp(prefix="l200-exam-"))

# 200 · Defend the result. Defend the next experiment.

<p class="eyebrow">Year 5 exit exam · 20-minute review + notebook + written defense</p>

**Your win:** produce one auditable FM reproduction and one research proposal that could prove your favorite explanation wrong. The mission is to establish when relational learning adds value. A close benchmark score is evidence for a specific computation; your proposal must explain what question remains.

<div class="exit-status"><strong>Fresh selected reproduction:</strong> COMPLETE_SELECTED_REPRODUCTION. 30 evaluations · 21,060 predictions. <strong>Year 5 exit:</strong> INCOMPLETE; proposal and defense pending. <strong>RDBLearn:</strong> INCOMPLETE_SOURCE_PREPROCESSING_GATE.</div>

## 1 · Close the books first



Write three answers before scrolling: What distinguishes released-checkpoint inference from pretraining? Why are ten support draws not ten independent replications? Why does an old event timestamp fail to prove that a feature was available?

<details><summary>Retrieve, then check</summary><p>Checkpoint inference uses existing learned weights. Pretraining learns those weights across tasks. The ten draws share one test population. Event time records when something happened; arrival time records when a predictor could access it. Neither score agreement nor an event timestamp establishes historical availability.</p></details>

**The bridge to this exam.** [L197](https://avistian.github.io/relational/lessons/0197-year-5-essay.html) assembled claims and limitations. [L198](https://avistian.github.io/relational/lessons/0198-three-research-directions.html) made three proposal cards; [L199](https://avistian.github.io/relational/lessons/0199-select-primary-direction.html) separated priority, feasibility and launch authorization. Now you must connect a measured result to one testable question. Bring your own cards and selection memo into this exam; the author's worked choice is an example to challenge, not a substitute for your decision.

If you have not finished those cards, complete the [self-contained worksheet](https://avistian.github.io/relational/labs/evidence/l200/proposal-template.md) here. Ranking candidates is authoring a plan; it does not establish novelty or learner mastery.

## 2 · State exactly what you reproduced

A **foundation model** here is a model pretrained across tasks and reused on a new task. An **in-context** prediction uses labeled support examples as input without updating the model's weights on the new task. A **checkpoint** is a saved set of learned weights. A **support draw** is a sampled set of training examples that the model may read.

Our named target is **RDB-PFN v5, Table 9, rel-f1/driver-dnf, 512 support examples**, with the relational-prior checkpoint, the single-table-prior checkpoint, and TabICL v1.1. Read the [primary paper and appendix](https://arxiv.org/html/2603.03805v5#A6), then the [frozen execution protocol](https://avistian.github.io/relational/labs/l200-reproduction.md).

| Held fixed | Full approved scope |
|---|---|
| Released rows | 11,411 train; 566 validation; 702 test |
| Support | 512 training rows without replacement, seeds 0–9 |
| Pairing | Each configuration receives the same support for a given draw |
| Checkpoints | Released fixed checkpoints; no test-based selection |
| Preprocessing | Support medians; all-missing becomes zero; support normalization |
| Evaluation | All 3 × 10 runs; all 702 queries per run |
| Comparison | Mean AUROC within 0.02 of the named rounded target |

**Why no task training loop?** RDB-PFN has already learned its weights in synthetic pretraining. The released evaluation places support features and labels beside query features. Calling the classifier's `fit` prepares that context; this lane does not optimize RDB-PFN weights on F1. Reproducing this experiment means performing fresh inference from the released weights. It does not mean repeating synthetic pretraining.

**Scope check.** The source checkpoint uses width 96; the paper appendix says 128. We preserve checkpoint-compatible width 96. The released binary label complements the currently reconstructed DNF label. Keep released labels and probabilities for table comparison; complement **both** for current DNF interpretation. Full raw feature reconstruction, historical identity and complete feature arrival provenance remain unresolved. Numerical closeness is not a leakage certificate. The separate RDBLearn reproduction remains `INCOMPLETE_SOURCE_PREPROCESSING_GATE`.

## 3 · Trace the model before trusting the score

**Predict:** can changing one query's hidden label change another query's output? It must not: query labels never enter the model, and row attention reads support rows only.

RDB-PFN information flow

**Inputs.** The released task has numeric DFS features. DFS means feature aggregation across related tables; those flat columns already contain relational information. The relational prior describes synthetic training tasks, not a graph traversed by this predictor at evaluation time. Both RDB-PFN checkpoints see the same released task features, so their difference is not a clean “graph versus no relations” comparison.

**Feature tokens.** Normalize each scalar feature with support statistics, clamp to ±100, then project it into 96 coordinates. Add a target token: observed support labels for support rows, the support-label mean for query rows. The tensor has shape `batch × rows × (features + 1) × 96`.

**Two attention axes.** Each of six blocks first mixes feature tokens within each row. It then mixes rows separately for each feature position. Every receiver reads keys and values from support rows only. Four attention heads divide the width into four 24-coordinate parts. Residual additions keep the previous representation; layer normalization rescales it. A `96 → 192 → 96` feed-forward network follows.

**Output.** A `96 → 192 → 2` decoder reads each query's target token. Softmax converts the two logits into class probabilities. We save the probability of released label 1 beside its complete `(driverId, date)` key. The [visible model source](https://avistian.github.io/relational/labs/evidence/l200/packet/rdbpfn_visible.py) is reproduced inline in the notebook. Its course prior helper is illustrative; it is not the released pretraining generator. Author verification compares the visible forward computation against both real checkpoint models in float64; the fresh full evaluator uses the unchanged float32 source path.

## 4 · Work the score by hand

**AUROC** is the fraction of positive–negative pairs in which the positive receives the higher score, with half credit for a tie. It measures ranking, not calibrated probabilities or operational benefit.

**Worked example.** Positive scores are `[0.5, 0.8]`; negative scores are `[0.1, 0.5]`. The four comparisons earn `1, 0.5, 1, 1`: AUROC is `3.5 / 4 = 0.875`. Ties matter. A row permutation should change nothing after joining predictions to labels by complete keys.

**Two keys, not one.** Driver 1 at cutoff 10 and driver 1 at cutoff 20 are different queries. Joining only on driver ID can silently attach the wrong outcome. The notebook asks you to implement the full-key join and reject duplicates, missing identities, nonfinite probabilities and incomplete run grids.



| Configuration | Fresh mean AUROC ± sample SD | Paper target | Comparison |
|---|---:|---:|---|
| RDBPFN | 0.721938 ± 0.019983 | 0.7219 | CLOSE |
| RDBPFN_single | 0.663990 ± 0.034177 | 0.6640 | CLOSE |
| TabICLv1.1 | 0.717568 ± 0.009770 | 0.7176 | CLOSE |

RDB-PFN minus TabICL: **+0.004369** mean paired difference, positive in **6/10** support draws. [Every draw and full report](https://avistian.github.io/relational/labs/evidence/l200/report.json). These are new L200 predictions.


The standard deviation describes variation over ten support draws on **one fixed test set**. It is not a confidence interval over databases. The tolerance of 0.02 is a predeclared descriptive acceptance rule, not a statistical equivalence test. A paired difference uses matching support draws; it still cannot establish broad superiority or an independent replication.

**Execution and agreement are separate.** All 30 valid runs can be complete while a target remains `OUTSIDE_TOLERANCE`. Report both. Never remove an awkward draw, switch label orientation on one side only, or use test performance to pick a checkpoint.

## 5 · A passing result does not pass the exam for you

Change each evidence state below. The baseline is a verified reproduction with proposal and defense pending. Notice that filling all fields only produces **ready for teacher review**. The widget is a planning aid; it neither grades your reasoning nor writes a mastery record.


<noscript><p>Baseline: reproduction PASS; proposal PENDING; defense PENDING → INCOMPLETE. All three PASS → READY_FOR_TEACHER_REVIEW. Any missing or failed requirement keeps the exam incomplete.</p></noscript>

The reproduction gate requires authenticated inputs, complete runs, independent scoring, comparison and a written account of deviations. The proposal gate requires a falsifiable question, controls, data availability, uncertainty and a complete cost estimate. The defense gate requires your explanation of why the evidence supports only the stated claim. These are distinct deliverables.

## 6 · Turn a result into a falsifiable proposal

**A hypothesis** is a statement that could be weakened by an observation. **A control** holds competing explanations steady. **A falsification rule** states beforehand which observation would make you revise the hypothesis. **A practical threshold**, δ, is the smallest effect worth acting on; justify it before collecting the proposed study's outcomes.

Complete three cards using the [worksheet](https://avistian.github.io/relational/labs/evidence/l200/proposal-template.md). For each, state the question, required evidence, baseline, cost and first blocking condition. Rank them with explicitly justified criteria, discuss sensitivity to those judgments, then select one. Do not copy L199's ordinal ratings onto L198's different experiments.

**Worked author example — not an approved new experiment.** Ask whether a model's apparent benefit from related-row context shrinks when features must have actually arrived by prediction time. In one fixed learner family, compare target-row-only input against added relational context under both event-time and measured-arrival policies. Keep labels, query identities, splits and tuning budget matched.

Let `D = (context − control)event − (context − control)arrival`, using the same higher-is-better metric in every cell. With illustrative AUROCs `0.75, 0.70, 0.71, 0.69`, `D = 0.05 − 0.02 = 0.03`. These four numbers are invented arithmetic, not experimental measurements. The formula asks how much the context advantage shrinks; it does not isolate graph architecture.

Predeclare δ and an uncertainty method appropriate to the experimental units. An interval entirely above δ supports practically meaningful shrinkage; an interval overlapping δ is inconclusive; an interval entirely below δ weakens that claim. Correlated support draws do not supply independent databases. The existing RDB-PFN score does not estimate D because it lacks the four matched information-policy conditions.

**First feasibility question:** can measured arrival histories be obtained and aligned to query cutoffs? If not, stop or explicitly redesign the question. Artificial delays would test a simulation claim. A budget estimate for future training is still required; this lesson's reproduction approval does not authorize that future study. Novelty remains `NOT_ESTABLISHED` until a relevant literature search and argument support it.

## 7 · Your submission and defense

Submit the notebook, complete run report with raw predictions and provenance, and a **1,000–1,500 word proposal**. Use this rubric; presence checks cannot grade scientific quality.

| Criterion | What a defensible submission shows | Points |
|---|---|---:|
| Reproduction | Exact target, full grid, pinned inputs, independent scores, deviations | 0–2 |
| Explanation | Support/query information flow, label semantics, evidence limits | 0–2 |
| Research design | One falsifiable hypothesis, matched controls, metric and δ | 0–2 |
| Feasibility | Available data, complete cost, stop rule and alternative | 0–2 |
| Defense | Counter-evidence, uncertainty, revision condition, own reasoning | 0–2 |

The teacher reviews all five criteria; a total alone cannot compensate for an invalid reproduction or unfalsifiable design. Required proposal sections: three alternatives, selected question, hypothesis, evidence, data/availability, baselines, splits, metric/threshold, uncertainty, full budget, stop rule and revision condition.



**Defense prompts.** Explain why a fresh checkpoint rerun is useful without being fresh pretraining. Identify one conclusion these scores cannot support. Describe one observation that would overturn your selected direction. Explain what remains missing before Year 6 execution.

**Exit state:** author artifacts can be verified while your Year 5 exit remains `INCOMPLETE` and `PENDING_WRITTEN_DEFENSE`. Earlier incomplete exits are not cleared here. After teacher review, proceed to the [Year 5 → 6 bridge](https://avistian.github.io/relational/reference/curriculum.html#research-bridge), updating the proposal as stronger baselines or architecture evidence warrant.

Revisit the three retrieval questions after **1, 7 and 30 days**. Ask the agent about any unclear step, or paste your proposal for rubric-based feedback. For practitioner feedback, prepare one bounded question with a minimal reproducer for the relevant project community; sending it is your choice.

**Resources:** [RDB-PFN v5](https://arxiv.org/html/2603.03805v5), [RelBench v2](https://arxiv.org/abs/2602.12606), [COS preregistration guidance](https://www.cos.io/initiatives/prereg), [protocol](https://avistian.github.io/relational/labs/l200-reproduction.md), [reference](https://avistian.github.io/relational/reference/year-5-exit-exam.html), [student notebook](https://avistian.github.io/relational/labs/0200-year-5-exit-exam.ipynb), [executed solution](https://avistian.github.io/relational/labs/html/0200-year-5-exit-exam.html).


In [2]:
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='17c0128e8cced6cd004c308878e9e506c4d4229298202b63051c69d16124f4fd'
with zipfile.ZipFile(io.BytesIO(payload)) as archive:archive.extractall(workspace)
evidence=workspace/"labs/evidence/l200"
print("Authenticated full raw packet",len(payload),"bytes")

Authenticated full raw packet 1487471 bytes


## PROVIDED · Visible checkpoint-compatible model

Trace the two attention axes, support statistics and target padding. This exact source defines the model below; it is not hidden inside an import. The two-table prior helper is explicitly a course illustration, not the released pretraining generator. The author performed checkpoint parity separately; the small live check here uses initialized weights.

In [3]:
"""Visible L166 mechanisms. Course prior is deliberately small; model matches base release."""
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

def relational_prior(seed, parents=8, children=24, strength=1.0):
    """Course two-table SCM; not the released LayerDAG/selective-SCM generator."""
    if any(type(v) is not int or v<1 for v in [parents,children]) or not np.isfinite(strength):
        raise ValueError('Positive integer sizes and finite strength required')
    rng=np.random.default_rng(seed)
    latent=rng.normal(size=parents)
    foreign_keys=rng.integers(parents,size=children)
    noise=rng.normal(scale=.2,size=children)
    return dict(parent_ids=np.arange(parents),latent=latent,child_parent=foreign_keys,
                noise=noise,values=strength*latent[foreign_keys]+noise)

def dfs_summary(parent_ids, child_parent, values):
    """One backward aggregation: COUNT and MEAN in the supplied parent order."""
    ids=list(parent_ids);fk=list(child_parent);v=np.asarray(values,dtype=float)
    if len(set(ids))!=len(ids) or len(fk)!=len(v) or v.ndim!=1 or not np.isfinite(v).all() or not set(fk)<=set(ids):
        raise ValueError('Unique parents, valid foreign keys and finite aligned values required')
    out=np.zeros((len(ids),2));lookup={key:i for i,key in enumerate(ids)}
    for key,value in zip(fk,v):
        out[lookup[key],0]+=1;out[lookup[key],1]+=value
    out[:,1]/=np.maximum(out[:,0],1)
    return out

def context_mask(rows, support):
    """True means a key/value is visible: all receivers read support only."""
    if type(rows) is not int or type(support) is not int or not 0<support<=rows:
        raise ValueError('Require 0 < support <= rows, both integers')
    return np.broadcast_to(np.arange(rows)<support,(rows,rows)).copy()

def normalize_support(x, support):
    """Released population-variance normalization before scalar-to-vector projection."""
    if x.ndim!=3 or not 0<support<=x.shape[1]:raise ValueError('B x N x F with nonempty support')
    z=x.unsqueeze(-1);train=z[:,:support];valid=~torch.isnan(train)
    count=valid.sum(1,keepdim=True).clamp(min=1)
    mean=torch.where(valid,train,0.).sum(1,keepdim=True)/count
    var=torch.where(valid,train-mean,0.).square().sum(1,keepdim=True)/count
    return ((torch.where(torch.isnan(z),mean,z)-mean)/torch.sqrt(var+1e-20)).clamp(-100,100)

class FeatureEncoder(nn.Module):
    def __init__(self,width):
        super().__init__();self.linear_layer=nn.Linear(1,width)
    def forward(self,x,support):return self.linear_layer(normalize_support(x,support))

class TargetEncoder(nn.Module):
    def __init__(self,width):
        super().__init__();self.linear_layer=nn.Linear(1,width)
    def forward(self,y,rows):
        # Query slots contain the support mean, never held-out query labels.
        padding=y.to(torch.float).mean(1,keepdim=True).repeat(1,rows-y.shape[1],1)
        return self.linear_layer(torch.cat([y,padding],1).unsqueeze(-1))

class BiAttention(nn.Module):
    def __init__(self,width,heads,hidden):
        super().__init__()
        self.self_attention_between_features=nn.MultiheadAttention(width,heads,batch_first=True)
        self.self_attention_between_datapoints=nn.MultiheadAttention(width,heads,batch_first=True)
        self.linear1=nn.Linear(width,hidden);self.linear2=nn.Linear(hidden,width)
        self.norm1=nn.LayerNorm(width);self.norm2=nn.LayerNorm(width);self.norm3=nn.LayerNorm(width)
    def forward(self,x,support):
        b,n,f,d=x.shape
        z=x.reshape(b*n,f,d)
        z=self.norm1((z+self.self_attention_between_features(z,z,z)[0]).reshape(b,n,f,d))
        z=z.transpose(1,2).reshape(b*f,n,d)
        keys=z[:,:support]
        left=self.self_attention_between_datapoints(keys,keys,keys)[0]
        right=self.self_attention_between_datapoints(z[:,support:],keys,keys)[0]
        z=self.norm2((z+torch.cat([left,right],1)).reshape(b,f,n,d).transpose(1,2))
        return self.norm3(z+self.linear2(F.gelu(self.linear1(z))))

class Decoder(nn.Module):
    def __init__(self,width,hidden):
        super().__init__();self.linear1=nn.Linear(width,hidden);self.linear2=nn.Linear(hidden,2)
    def forward(self,x):return self.linear2(F.gelu(self.linear1(x)))

class RDBPFN(nn.Module):
    """Checkpoint-compatible numeric base: six blocks, 96 channels, four heads."""
    def __init__(self,width=96,heads=4,hidden=192,layers=6):
        super().__init__();self.feature_encoder=FeatureEncoder(width);self.target_encoder=TargetEncoder(width)
        self.transformer_blocks=nn.ModuleList([BiAttention(width,heads,hidden) for _ in range(layers)])
        self.decoder=Decoder(width,hidden)
    def forward(self,src,support):
        x,y=src
        if y.ndim==2:y=y.unsqueeze(-1)
        z=torch.cat([self.feature_encoder(x,support),self.target_encoder(y,x.shape[1])],2)
        for block in self.transformer_blocks:z=block(z,support)
        return self.decoder(z[:,support:,-1])


In [4]:
torch.manual_seed(200)
model=RDBPFN().double().eval()
x=torch.randn(1,7,3,dtype=torch.float64)
y=torch.tensor([[0.,1.,0.,1.]],dtype=torch.float64)
with torch.no_grad():
    baseline=model((x,y),4)
    changed=x.clone();changed[:,5,:]+=10
    intervened=model((changed,y),4)
assert baseline.shape==(1,3,2)
torch.testing.assert_close(baseline[:,0],intervened[:,0],atol=1e-10,rtol=1e-10)
print("Changing another query leaves the first query unchanged; query labels are absent.")

Changing another query leaves the first query unchanged; query labels are absent.


## TODO · keyed_auc

**Goal:** Join two-column complete keys, reject duplicates/missing keys and invalid labels/probabilities; return tie-aware pairwise AUROC. Binary labels must contain both classes.

**Why:** incorrect evidence admission can invalidate an apparently close score. **Hint:** validate before aggregating; do not edit CHECK.

In [5]:
def keyed_auc(keys, labels, prediction_keys, probabilities):
    """Join on complete entity/cutoff keys; AUROC uses half credit for ties."""
    k=np.asarray(keys);pk=np.asarray(prediction_keys);y=np.asarray(labels);p=np.asarray(probabilities,dtype=float)
    if k.ndim!=2 or k.shape[1]!=2 or pk.shape!=k.shape or y.shape!=(len(k),) or p.shape!=y.shape:
        raise ValueError('Invalid shapes')
    a=list(map(tuple,k.tolist()));b=list(map(tuple,pk.tolist()))
    if len(set(a))!=len(a) or len(set(b))!=len(b) or set(a)!=set(b):raise ValueError('Missing or duplicate full key')
    if not np.isfinite(p).all() or np.any((p<0)|(p>1)) or set(y.tolist())!={0,1}:raise ValueError('Invalid label/probability')
    lookup=dict(zip(b,p));ordered=np.array([lookup[key] for key in a]);pos=ordered[y==1];neg=ordered[y==0]
    return float(((pos[:,None]>neg).sum()+.5*(pos[:,None]==neg).sum())/(len(pos)*len(neg)))

## TODO · complete_grid

**Goal:** Require precisely three named arms × seeds0–9,702rows and512support each. Reject booleans as numeric counts. Return runs=30,predictions=21060 only for the complete grid.

**Why:** incorrect evidence admission can invalidate an apparently close score. **Hint:** validate before aggregating; do not edit CHECK.

In [6]:
def complete_grid(records):
    """Require exactly 3 configurations × 10 draws, never summarize a partial grid."""
    expected={(a,s) for a in ['RDBPFN','RDBPFN_single','TabICLv1.1'] for s in range(10)};seen=set()
    for row in records:
        if not isinstance(row,dict) or any(k not in row for k in ['arm','seed','rows','support']):raise ValueError('Missing fields')
        if not isinstance(row['arm'],str) or any(type(row[k]) is not int for k in ['seed','rows','support']):raise ValueError('Invalid field type')
        identity=(row['arm'],row['seed'])
        if identity not in expected or identity in seen or row['rows']!=702 or row['support']!=512:raise ValueError('Wrong run grid')
        seen.add(identity)
    if seen!=expected:raise ValueError('Incomplete grid')
    return dict(runs=len(seen),predictions=702*len(seen))

## TODO · exit_gate

**Goal:** Accept PASS/FAIL/PENDING only. Return ordered blockers and INCOMPLETE if any is not PASS; otherwise READY_FOR_TEACHER_REVIEW. Never grant mastery.

**Why:** incorrect evidence admission can invalidate an apparently close score. **Hint:** validate before aggregating; do not edit CHECK.

In [7]:
def exit_gate(reproduction, proposal, defense):
    """Author verification cannot fill the learner's proposal or oral/written defense."""
    values=dict(reproduction=reproduction,proposal=proposal,defense=defense)
    if any(v not in ['PASS','FAIL','PENDING'] for v in values.values()):raise ValueError('Unknown state')
    blockers=[k for k,v in values.items() if v!='PASS']
    return dict(state='INCOMPLETE' if blockers else 'READY_FOR_TEACHER_REVIEW',blockers=blockers)

## CHECK · live learner contracts

These call your implementations. A constant score, accepting a partial grid, or automatic exam pass fails.

In [8]:
def checks(score, grid, gate):
    keys=[[1,10],[1,20],[2,10],[2,20]]
    assert score(keys,[0,1,0,1],keys[::-1],[.8,.5,.5,.1])==.875
    def rejects(fn,*args):
        try:fn(*args)
        except ValueError:return
        raise AssertionError('Invalid evidence accepted')
    rejects(score,keys,[0,1,0,1],keys[:-1],[.1,.2,.3])
    rejects(score,keys,[0,1,0,1],[keys[0]]*4,[.1,.2,.3,.4])
    rejects(score,keys,[0,1,0,1],keys,[.1,float('nan'),.3,.4])
    rejects(score,keys,[0,0,0,0],keys,[.1,.2,.3,.4])
    rejects(score,keys,[0,1,0,1],keys,[.1,1.2,.3,.4])
    rows=[dict(arm=a,seed=s,rows=702,support=512) for a in ['RDBPFN','RDBPFN_single','TabICLv1.1'] for s in range(10)]
    assert grid(rows)==dict(runs=30,predictions=21060)
    rejects(grid,rows[:-1]);rejects(grid,rows+[rows[0]])
    changed=[dict(r) for r in rows];changed[0]['rows']=701;rejects(grid,changed)
    changed=[dict(r) for r in rows];changed[0]['seed']=False;rejects(grid,changed)
    assert gate('PASS','PENDING','PENDING')==dict(state='INCOMPLETE',blockers=['proposal','defense'])
    assert gate('PASS','PASS','PASS')==dict(state='READY_FOR_TEACHER_REVIEW',blockers=[])
    assert gate('FAIL','PASS','PASS')['state']=='INCOMPLETE'
    rejects(gate,'COMPLETE','PASS','PASS')
    return 'PASS'
print(checks(keyed_auc,complete_grid,exit_gate))

PASS


## PROVIDED · Complete authenticated report

The report below joins each fresh prediction with the original query labels and supports. Your functions are passed into the actual report computation. All30runs are required.

In [9]:
def audit(root,score=keyed_auc,grid=complete_grid,gate=exit_gate):
    root=Path(root);manifest=json.loads((root/'packet-manifest.json').read_text())
    for name,h in manifest['files'].items():
        if hashlib.sha256((root/name).read_bytes()).hexdigest()!=h:raise ValueError('Changed input '+name)
    data=np.load(root/'packet/prepared.npz');records=[];runs=[];vectors={a:[] for a in ['RDBPFN','RDBPFN_single','TabICLv1.1']}
    for phase in ['pilot-1','full-1']:
        receipt=json.loads((root/phase/'receipt.json').read_text())
        if receipt['input_manifest_sha256']!=manifest['input_manifest_sha256']:raise ValueError('Wrong input receipt')
        for row in receipt['records']:
            records.append(row);path=root/phase/f"{row['arm']}-{row['seed']}.npz"
            if hashlib.sha256(path.read_bytes()).hexdigest()!=row['sha256']:raise ValueError('Changed predictions')
            x=np.load(path);seed=row['seed']
            np.testing.assert_array_equal(x['keys'],data['test_keys']);np.testing.assert_array_equal(x['label'],data['y_test'])
            np.testing.assert_array_equal(x['support_keys'],data['train_keys'][data['support'][seed]])
            if len(set(map(tuple,x['support_keys'])))!=512 or set(map(tuple,x['keys'])) & set(map(tuple,x['support_keys'])):raise ValueError('Invalid support identities')
            auc=score(data['test_keys'],data['y_test'],x['keys'],x['probability'])
            if abs(auc-row['auc'])>1e-12:raise ValueError('Receipt score mismatch')
            runs.append(dict(arm=row['arm'],seed=seed,auc=auc))
    counts=grid(records)
    for row in sorted(runs,key=lambda r:(r['arm'],r['seed'])):vectors[row['arm']].append(row['auc'])
    targets=dict(RDBPFN=.7219,RDBPFN_single=.6640,**{'TabICLv1.1':.7176})
    models={a:dict(mean=float(np.mean(v)),sample_sd=float(np.std(v,ddof=1)),per_seed=v,paper_target=targets[a],status='CLOSE' if abs(np.mean(v)-targets[a])<=.02 else 'OUTSIDE_TOLERANCE') for a,v in vectors.items()}
    deltas=np.array(vectors['RDBPFN'])-vectors['TabICLv1.1']
    return dict(experiment='L200 RDB-PFN v5 Table9 F1 driver-dnf 512-support fresh checkpoint evaluation',execution='COMPLETE_SELECTED_REPRODUCTION',**counts,models=models,paired_rdbpfn_minus_tabicl=dict(mean=float(deltas.mean()),sample_sd=float(deltas.std(ddof=1)),positive=int((deltas>0).sum()),per_seed=deltas.tolist()),comparison='CLOSE' if all(m['status']=='CLOSE' for m in models.values()) else 'OUTSIDE_TOLERANCE',tolerance=.02,exam=gate('PASS' if all(m['status']=='CLOSE' for m in models.values()) else 'FAIL','PENDING','PENDING'),learner='PENDING_WRITTEN_DEFENSE',rdb_learn='INCOMPLETE_SOURCE_PREPROCESSING_GATE',fresh_pretraining='NOT_RUN',whole_paper='NOT_RUN',historical_identity='NOT_ESTABLISHED',full_feature_availability='NOT_ESTABLISHED',proposal_source='Frozen L198 and L199 author reports; learner submission required')
report=audit(evidence,score=keyed_auc,grid=complete_grid,gate=exit_gate)
assert report==json.loads((evidence/"report.json").read_text())
Path("l200-report.json").write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))

{
  "experiment": "L200 RDB-PFN v5 Table9 F1 driver-dnf 512-support fresh checkpoint evaluation",
  "execution": "COMPLETE_SELECTED_REPRODUCTION",
  "runs": 30,
  "predictions": 21060,
  "models": {
    "RDBPFN": {
      "mean": 0.7219377348362855,
      "sample_sd": 0.019982667847990538,
      "per_seed": [
        0.7383984775289123,
        0.7175523349436392,
        0.7038598545844923,
        0.7255941053042503,
        0.6823598301859172,
        0.7242277850973503,
        0.7217342507197579,
        0.7440491875274484,
        0.7114917288830332,
        0.7501097935880544
      ],
      "paper_target": 0.7219,
      "status": "CLOSE"
    },
    "RDBPFN_single": {
      "mean": 0.6639896550041476,
      "sample_sd": 0.03417677717319856,
      "per_seed": [
        0.6116234811886986,
        0.687180988630264,
        0.6655638510710975,
        0.6885375494071146,
        0.6797443029327087,
        0.6583711511247743,
        0.6154686966281169,
        0.6305274971941639,
 

## PROVIDED · Independent verifier

This second AUROC algorithm sorts and assigns tied average ranks. It does not reuse pairwise comparisons. The full verifier also rejects seven evidence mutations in disposable copies.

In [10]:
def rank_auc(y,p):
    # Independent of pairwise scorer: sort, average tied ranks, Mann-Whitney statistic.
    y=np.asarray(y);p=np.asarray(p);order=np.argsort(p,kind='stable');ranks=np.empty(len(p));i=0
    while i<len(p):
        j=i+1
        while j<len(p) and p[order[j]]==p[order[i]]:j+=1
        ranks[order[i:j]]=(i+1+j)/2;i=j
    positives=int(y.sum());negatives=len(y)-positives
    return float((ranks[y==1].sum()-positives*(positives+1)/2)/(positives*negatives))

def verify(e):
    e=Path(e);report=audit(e);assert checks(keyed_auc,complete_grid,exit_gate)=='PASS';errors=[]
    for phase in ['pilot-1','full-1']:
        receipt=json.loads((e/phase/'receipt.json').read_text())
        for row in receipt['records']:
            x=np.load(e/phase/f"{row['arm']}-{row['seed']}.npz")
            value=rank_auc(x['label'],x['probability']);errors.append(abs(value-row['auc']));assert abs(value-row['auc'])<1e-12
    rng=np.random.default_rng(200)
    for i in range(100):
        y=np.r_[0,1,rng.integers(0,2,18)];p=rng.integers(0,8,20)/7;k=np.array([[j%5,j//5] for j in range(20)]);order=rng.permutation(20)
        assert abs(rank_auc(y,p)-keyed_auc(k,y,k[order],p[order]))<1e-12
    wrong=[(lambda *x:.5,complete_grid,exit_gate),(keyed_auc,lambda x:dict(runs=30,predictions=21060),exit_gate),(keyed_auc,complete_grid,lambda *x:dict(state='READY_FOR_TEACHER_REVIEW',blockers=[]))]
    for fs in wrong:
        try:checks(*fs)
        except (AssertionError,ValueError):pass
        else:raise AssertionError('Wrong learner function accepted')
    # Mutations happen only in disposable copies. Hash corruption and semantically
    # corrupt but rehashed records must both fail; never edit real predictions.
    rejected=[]
    for mode in ['bytes','missing_run','duplicate_run','keys','supports','labels','probability']:
        with tempfile.TemporaryDirectory() as td:
            q=Path(td)/'evidence';shutil.copytree(e,q)
            path=q/'pilot-1/RDBPFN-0.npz';receipt_path=q/'pilot-1/receipt.json'
            receipt=json.loads(receipt_path.read_text())
            if mode=='bytes':path.write_bytes(path.read_bytes()+b'corrupt')
            elif mode in ['missing_run','duplicate_run']:
                if mode=='missing_run':receipt['records'].pop()
                else:receipt['records'].append(receipt['records'][0])
                receipt_path.write_text(json.dumps(receipt))
            else:
                with np.load(path) as x:arrays={k:x[k].copy() for k in x.files}
                if mode=='keys':arrays['keys'][0]=arrays['keys'][1]
                if mode=='supports':arrays['support_keys'][0]=arrays['support_keys'][1]
                if mode=='labels':arrays['label'][0]=1-arrays['label'][0]
                if mode=='probability':arrays['probability'][0]=np.nan
                np.savez_compressed(path,**arrays);receipt['records'][0]['sha256']=hashlib.sha256(path.read_bytes()).hexdigest();receipt_path.write_text(json.dumps(receipt))
            if mode!='bytes':
                manifest=json.loads((q/'packet-manifest.json').read_text())
                for name in manifest['files']:manifest['files'][name]=hashlib.sha256((q/name).read_bytes()).hexdigest()
                (q/'packet-manifest.json').write_text(json.dumps(manifest))
            try:audit(q)
            except (ValueError,AssertionError):rejected.append(mode)
            else:raise AssertionError('Corruption accepted '+mode)
    return dict(status='PASS',independent_rank_auc_runs=len(errors),maximum_error=max(errors),randomized_keyed_cases=100,wrong_learner_functions_rejected=3,corruptions_rejected=rejected,report_parity=report==json.loads((e/'report.json').read_text()))
print(verify(evidence))

{'status': 'PASS', 'independent_rank_auc_runs': 30, 'maximum_error': 1.1102230246251565e-16, 'randomized_keyed_cases': 100, 'wrong_learner_functions_rejected': 3, 'corruptions_rejected': ['bytes', 'missing_run', 'duplicate_run', 'keys', 'supports', 'labels', 'probability'], 'report_parity': True}


## EXIT · Write and defend your own proposal

Use the rubric and template above. Blank or author examples are not learner evidence. Save your1,000–1,500word proposal separately and bring it to the teacher. The following record deliberately keeps both learner gates pending.

In [11]:
submission={"reproduction":report["comparison"],"proposal":"PENDING_WRITTEN_DEFENSE","defense":"PENDING_WRITTEN_DEFENSE","exam":exit_gate("PASS" if report["comparison"]=="CLOSE" else "FAIL","PENDING","PENDING")}
Path("l200-submission.json").write_text(json.dumps(submission,indent=2))
Path("l200-proposal-template.md").write_text((evidence/"proposal-template.md").read_text())
assert time.monotonic()-started<3600
print(submission)

{'reproduction': 'CLOSE', 'proposal': 'PENDING_WRITTEN_DEFENSE', 'defense': 'PENDING_WRITTEN_DEFENSE', 'exam': {'state': 'INCOMPLETE', 'blockers': ['proposal', 'defense']}}


## Optional · Full fresh checkpoint execution

The author already executed all30fresh evaluations for L200. To run again, use a new output directory and a separate execution budget. Python3.11 with pinned [requirements](https://avistian.github.io/relational/labs/l166-requirements.txt) and Torch2.5.1; download the course repository, authenticate [protocol](https://avistian.github.io/relational/labs/l200-reproduction.md) and source ledger. This may require GPU resources; do not run it as part of the default notebook. No automatic download or paid dispatch occurs here.

```sh
python labs/_fetch_l166.py --out /tmp/l200-input
python labs/_run_l166.py --input /tmp/l200-input --out /tmp/l200-fresh-evaluation
```

The next cell exposes the exact complete evaluator source for inspection. It is stored as text, not executed in this replay session.

In [12]:
fresh_evaluator_source = '"""Complete selected released-checkpoint evaluation; no checkpoint search or fitting."""\nimport os\nos.environ.update(OMP_NUM_THREADS=\'2\',OPENBLAS_NUM_THREADS=\'2\',MKL_NUM_THREADS=\'2\',HF_HUB_OFFLINE=\'1\')\nimport hashlib,json,sys,time,random\nfrom pathlib import Path\nimport numpy as np\nimport torch\n\ndef run166(root,source,out,seeds):\n    from importlib.metadata import version\n    root=Path(root);out=Path(out);out.mkdir(parents=True,exist_ok=True)\n    manifest=json.loads((root/\'input-manifest.json\').read_text())\n    for name,entry in manifest[\'files\'].items():\n        assert hashlib.sha256((root/name).read_bytes()).hexdigest()==entry[\'sha256\'],name\n    sys.path.insert(0,str(Path(source)/\'model_pretrain\'))\n    from src.models import ModelConfig,build_model,load_checkpoint,build_classifier\n    from src.eval_utils import fill_nans,predict_proba_in_chunks\n    from sklearn.metrics import roc_auc_score\n    from tabicl import TabICLClassifier\n    assert version(\'tabicl\')==\'0.1.3\'\n    torch.set_num_threads(2)\n    device=torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    data=np.load(root/\'prepared.npz\');records=[]\n    for arm in [\'RDBPFN\',\'RDBPFN_single\',\'TabICLv1.1\']:\n        random.seed(42);np.random.seed(42);torch.manual_seed(42)\n        if torch.cuda.is_available():torch.cuda.manual_seed_all(42)\n        start=time.perf_counter()\n        if arm!=\'TabICLv1.1\':\n            cfg=ModelConfig(num_layers=6);model=build_model(cfg)\n            load_checkpoint(model,root/(arm+\'.pt\'),str(device));model.to(device).eval()\n        load_seconds=time.perf_counter()-start\n        for seed in seeds:\n            path=out/f\'{arm}-{seed}.npz\'\n            assert not path.exists(),\'Immutable attempt already exists\'\n            start=time.perf_counter();idx=data[\'support\'][seed]\n            X,Q=fill_nans(data[\'X_train\'][idx],data[\'X_test\']);y=data[\'y_train\'][idx]\n            clf=(build_classifier(model,device,cfg) if arm!=\'TabICLv1.1\' else\n                 TabICLClassifier(model_path=str(root/\'tabicl-classifier-v1.1-0506.ckpt\'),allow_auto_download=False,device=str(device)))\n            assert arm!=\'TabICLv1.1\' or clf.n_estimators==32\n            if torch.cuda.is_available():torch.cuda.reset_peak_memory_stats()\n            clf.fit(X,y);prob=predict_proba_in_chunks(clf,Q,2000)\n            assert prob.shape==(702,2) and np.isfinite(prob).all()\n            np.testing.assert_allclose(prob.sum(1),1,atol=1e-5)\n            np.savez_compressed(path,keys=data[\'test_keys\'],label=data[\'y_test\'],probability=prob[:,1],support_keys=data[\'train_keys\'][idx])\n            record=dict(arm=arm,seed=int(seed),rows=len(Q),support=len(idx),auc=float(roc_auc_score(data[\'y_test\'],prob[:,1])),seconds=time.perf_counter()-start,\n                        load_seconds=load_seconds,peak_gpu_bytes=torch.cuda.max_memory_allocated() if torch.cuda.is_available() else 0,\n                        sha256=hashlib.sha256(path.read_bytes()).hexdigest())\n            records.append(record);(out/f\'{arm}-{seed}.json\').write_text(json.dumps(record,indent=2)+\'\\n\');print(json.dumps(record),flush=True)\n        if arm!=\'TabICLv1.1\':del model\n        del clf\n        if torch.cuda.is_available():torch.cuda.empty_cache()\n    receipt=dict(records=records,device=str(device),packages={n:version(n) for n in [\'torch\',\'numpy\',\'pandas\',\'scikit-learn\',\'pydantic\',\'tabicl\']},input_manifest_sha256=hashlib.sha256((root/\'input-manifest.json\').read_bytes()).hexdigest())\n    (out/\'receipt.json\').write_text(json.dumps(receipt,indent=2)+\'\\n\');return receipt\n\nif __name__==\'__main__\':\n    import argparse\n    parser=argparse.ArgumentParser();parser.add_argument(\'--input\',default=\'/tmp/l166-input\');parser.add_argument(\'--source\',default=str(Path(__file__).parent/\'sources/l166/upstream\'));parser.add_argument(\'--out\',required=True);parser.add_argument(\'--seeds\',default=\'0,1,2,3,4,5,6,7,8,9\');a=parser.parse_args()\n    run166(a.input,a.source,a.out,[int(v) for v in a.seeds.split(\',\')])\n'
print(fresh_evaluator_source)

"""Complete selected released-checkpoint evaluation; no checkpoint search or fitting."""
import os
os.environ.update(OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',MKL_NUM_THREADS='2',HF_HUB_OFFLINE='1')
import hashlib,json,sys,time,random
from pathlib import Path
import numpy as np
import torch

def run166(root,source,out,seeds):
    from importlib.metadata import version
    root=Path(root);out=Path(out);out.mkdir(parents=True,exist_ok=True)
    manifest=json.loads((root/'input-manifest.json').read_text())
    for name,entry in manifest['files'].items():
        assert hashlib.sha256((root/name).read_bytes()).hexdigest()==entry['sha256'],name
    sys.path.insert(0,str(Path(source)/'model_pretrain'))
    from src.models import ModelConfig,build_model,load_checkpoint,build_classifier
    from src.eval_utils import fill_nans,predict_proba_in_chunks
    from sklearn.metrics import roc_auc_score
    from tabicl import TabICLClassifier
    assert version('tabicl')=='0.1.3'
    torch.set_num_